# Gate-Specific Template Encoding Report

This notebook evaluates common-subexpression reuse for reusable XOR/AND Tseitin templates. It is a standalone prototype for future integration into comput2cnf/pauli2cnf; it does not modify the generated encoders.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'benchmarks').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'benchmarks' / 'results' / 'template_encoding.csv'
if not RESULTS.exists():
    raise FileNotFoundError(f'Run benchmarks/template_encoding_benchmark.py first: {RESULTS}')

with RESULTS.open(encoding='utf-8', newline='') as stream:
    rows = list(csv.DictReader(stream))
len(rows), RESULTS

In [ ]:
for operation in sorted({row['operation'] for row in rows}):
    print(operation)
    for repeats in (1, 8, 64):
        subset = [row for row in rows if row['operation'] == operation and int(row['repeats']) == repeats]
        print(repeats, [(row['cache'], row['variables'], row['clauses'], row['cache_hits']) for row in subset])

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, operation, metric, title in [
    (axes[0], 'cnot', 'variables', 'Variables'),
    (axes[1], 'toffoli', 'clauses', 'Clauses'),
]:
    for cache, color in [('off', '#bdbdbd'), ('on', '#2a6fbb')]:
        subset = [row for row in rows if row['operation'] == operation and row['cache'] == cache]
        subset.sort(key=lambda row: int(row['repeats']))
        axis.plot([int(row['repeats']) for row in subset], [int(row[metric]) for row in subset], marker='o', color=color, label=f'cache {cache}')
    axis.set_xscale('log', base=2)
    axis.set_title(f'{operation.upper()} template: {title}')
    axis.set_xlabel('Repeated identical relation')
    axis.set_ylabel(title)
    axis.grid(alpha=0.25)
    axis.legend()
figure.suptitle('Common-subexpression reuse in gate-specific templates')
figure.tight_layout()
plt.show()

## Interpretation

The cache is beneficial when the same Boolean relation is reused with the same input literals. In a future Quokka-Sharp integration, the cache must be scoped carefully because different circuit time steps generally have different state variables. The current result supports applying this technique inside a gate template or a fused local block, not blindly across all circuit wires.